# ARC Prize 2026 — Entrenamiento LoRA (kernel de Kaggle, GPU L4×4)

Corre en GPU de Kaggle **L4×4 (96GB, internet OFF obligatorio)**. Pipeline: construir dataset
augmentado → fine-tune SFT con LoRA → guardar adaptador en `/kaggle/working` (luego se publica
como dataset para el kernel de inferencia `submit.ipynb`).

**Adjuntar a este kernel (todo offline):**
- La competencia `arc-prize-2026-arc-agi-2` (datos).
- El código del repo como dataset (con `src/` y `scripts/`).
- El **modelo base** desde Kaggle Models (p.ej. Qwen2.5-7B-Instruct) — NO se descarga de HF.

> L4×4 exige internet deshabilitado y consume quota al doble; solo disponible adjuntado a esta competencia.

In [ ]:
# OFFLINE (L4x4 exige internet OFF): nada de pip install ni descargas HF.
# La imagen GPU de Kaggle trae torch/transformers/peft/accelerate (NO bitsandbytes!).
import os, sys, glob, json, zipfile
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

OUT_DIR = '/kaggle/working/lora-adapter'
DATA_JSONL = '/kaggle/working/finetune_train.jsonl'

# RUN REAL: Qwen2.5-3B fp16 en las 2 T4 via DDP (un proceso por GPU).
SMOKE = False
N_AUG = 2 if SMOKE else 4
N_SYNTH = 200 if SMOKE else 8000
EPOCHS = 2.0
BATCH = 1
GRAD_ACCUM = 8                       # x2 procesos DDP -> batch efectivo 16
MAX_SEQ_LEN = 1536
MAX_STEPS = 40 if SMOKE else 3000
SAVE_STEPS = 400
NUM_GPUS = 2

print('INPUTS:', glob.glob('/kaggle/input/*'))

# Localiza el codigo (src/arc) en cualquier nivel; si viene como zips, los extrae
arc_init = glob.glob('/kaggle/input/**/src/arc/__init__.py', recursive=True)
if not arc_init:
    dst = '/kaggle/working/code'; os.makedirs(dst, exist_ok=True)
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        if os.path.basename(z) in ('src.zip', 'scripts.zip'):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(os.path.join(dst, os.path.basename(z)[:-4]))
    arc_init = glob.glob(dst + '/**/arc/__init__.py', recursive=True)
assert arc_init, 'No se encontro src/arc. Adjunta el dataset de codigo (arc-agi2-code).'
SRC_DIR = os.path.dirname(os.path.dirname(arc_init[0]))
sys.path.insert(0, SRC_DIR)
sc = glob.glob('/kaggle/**/scripts/build_finetune_dataset.py', recursive=True)
assert sc, 'No se encontro scripts/ en el dataset de codigo.'
SCRIPTS_DIR = os.path.dirname(sc[0]); sys.path.insert(0, SCRIPTS_DIR)
TRAIN_SCRIPT = os.path.join(SCRIPTS_DIR, 'train_lora.py')

# Datos de la competencia
ch = glob.glob('/kaggle/input/**/arc-agi_training_challenges.json', recursive=True)
assert ch, 'No se encontraron los datos de la competencia.'
COMP_DIR = os.path.dirname(ch[0])

# MODELO BASE: dir con config.json + pesos (safetensors/bin) y SIN adapter_config.json
MODEL_DIR = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg)
    files = os.listdir(d)
    if any(f.endswith(('.safetensors', '.bin')) for f in files) and 'adapter_config.json' not in files:
        MODEL_DIR = d; break
assert MODEL_DIR, 'Adjunta el modelo base (Kaggle Models) — Qwen2.5-3B-Instruct.'
print('SMOKE =', SMOKE, '| SRC_DIR =', SRC_DIR, '| TRAIN_SCRIPT =', TRAIN_SCRIPT)
print('COMP_DIR =', COMP_DIR); print('MODEL_DIR =', MODEL_DIR)

In [ ]:
# 1) Construir dataset = tareas REALES augmentadas + tareas SINTÉTICAS
from arc.data import load_split
from arc.synth import generate_dataset
from build_finetune_dataset import samples_for_task

n = 0
with open(DATA_JSONL, 'w', encoding='utf-8') as f:
    # reales (D4 + color + leave-one-out)
    for task in load_split('training', data_dir=COMP_DIR).values():
        for s in samples_for_task(task, n_aug=N_AUG, max_cells=900, sep=''):
            f.write(json.dumps(s) + '\n'); n += 1
    # sintéticas (la regla es consistente por tarea; el modelo aprende a inducirla)
    for task in generate_dataset(N_SYNTH, seed=0):
        for s in samples_for_task(task, n_aug=1, max_cells=900, sep=''):
            f.write(json.dumps(s) + '\n'); n += 1
print('Muestras totales:', n, '->', DATA_JSONL)

In [ ]:
# 2) Fine-tune SFT con LoRA via DDP (un proceso por GPU) con accelerate launch.
import subprocess
cmd = [
    sys.executable, '-m', 'accelerate.commands.launch',
    '--multi_gpu', '--num_processes', str(NUM_GPUS), '--mixed_precision', 'fp16',
    TRAIN_SCRIPT,
    '--data', DATA_JSONL, '--model', MODEL_DIR, '--out', OUT_DIR,
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH), '--grad-accum', str(GRAD_ACCUM),
    '--max-seq-len', str(MAX_SEQ_LEN), '--max-steps', str(MAX_STEPS),
    '--save-steps', str(SAVE_STEPS), '--lora-r', '16', '--device-map', 'none',
]
env = dict(os.environ)
env['PYTHONPATH'] = SRC_DIR + os.pathsep + SCRIPTS_DIR
print('CMD:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
assert r.returncode == 0, f'entrenamiento DDP fallo rc={r.returncode}'

In [ ]:
# 3) Verifica el adaptador guardado por el entrenamiento DDP
import os
print('OUT_DIR:', os.listdir(OUT_DIR) if os.path.isdir(OUT_DIR) else 'NO EXISTE')
assert os.path.exists(os.path.join(OUT_DIR, 'adapter_model.safetensors')), 'no se guardo el adaptador'
print('OK: adapter_model.safetensors presente')

## Tras ejecutar
1. Guarda la salida del kernel (`/kaggle/working/lora-adapter`) como **Dataset** (p.ej. `arc-agi2-model`).
2. Adjunta ese dataset (y el modelo base) al kernel de inferencia `submit.ipynb` (internet OFF).
3. Valida en `evaluation` con `scripts/evaluate.py` antes de enviar.